<a href="https://colab.research.google.com/github/daviseemann/turbofan-rul-prediction-cmapss/blob/production/notebooks/MLP-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

drive.mount(
    "/content/drive/",
)
import os

os.chdir(
    "/content/drive/MyDrive/Data science studies/Aprendizado-de-maquina-UFSC/final-project/data"
)

In [0]:
# Caminhos dos arquivos
train_path = "./6.turbofan rul/train_FD001.txt"
test_path = "./6.turbofan rul/test_FD001.txt"
rul_path = "./6.turbofan rul/RUL_FD001.txt"

# Nomes das colunas (de acordo com a documentação original do C-MAPSS)
column_names = (
    ["engine_id", "cycle"]
    + [f"op{i}" for i in range(1, 4)]
    + [f"s{i}" for i in range(1, 22)]
)

# Importando os arquivos (espaço em branco como delimitador)
df_train_raw = pd.read_csv(train_path, sep="\s+", header=None, names=column_names)
df_test = pd.read_csv(test_path, sep="\s+", header=None, names=column_names)
df_rul = pd.read_csv(rul_path, sep="\s+", header=None, names=["RUL"])

In [0]:
display(df_train_raw.head())
display(df_test.head())
display(df_rul.head())

In [0]:
def plots(history, xlim=None, ylim=None):
  if hasattr(history, 'history'):
    history = history.history
  plt.figure(figsize=(14, 4))
  plt.subplot(1, 2, 1)
  plt.plot(history['loss'], '.-', label='Train loss')
  if 'val_loss' in history.keys():
    plt.plot(history['val_loss'], '.-', label='Val loss')
  plt.xlabel('Epochs');
  plt.legend();
  plt.yscale('log')
  plt.grid(which='both');
  plt.subplot(1, 2, 2)
  plt.plot(history['rmse'], '.-', label='Train rmse')
  plt.xlabel('Epochs');
  if 'val_rmse' in history.keys(): # Check for val_rmse explicitly
    plt.plot(history['val_rmse'], '.-', label='Val rmse')
    # Correct title to show actual RMSE values and best (minimum) val_rmse
    plt.title(f"Val rmse: {np.min(history['val_rmse']):.2f} (best) | {history['val_rmse'][-1]:.2f} (last)");
  # if 'val_rul_health_score' in history.keys(): # Check for val_rmse explicitly
  #   plt.plot(history['val_rul_health_score'], '.-', label='val rhs')
  #   # Correct title to show actual RMSE values and best (minimum) val_rmse
  #   plt.title(f"Val rhs: {np.min(history['val_rul_health_score']):.2f} (best) | {history['val_rul_health_score'][-1]:.2f} (last)");
  plt.legend();
  plt.xlim(xlim);
  plt.ylim(ylim);
  plt.grid();

# Pré-processamento


### Seleção de Sensores

O artigo menciona que apenas 14 dos 21 sensores são usados. Vamos selecionar os mesmos:


In [0]:
# Sensores selecionados conforme o artigo
selected_sensors = [
    "s2",
    "s3",
    "s4",
    "s7",
    "s8",
    "s9",
    "s11",
    "s12",
    "s13",
    "s14",
    "s15",
    "s17",
    "s20",
    "s21",
]

# Colunas que vamos manter
features_to_keep = ["engine_id", "cycle"] + selected_sensors

# Filtrando os dataframes
df_train_raw = df_train_raw[features_to_keep]
df_test = df_test[features_to_keep]

### Normalização dos Dados

O artigo usa normalização min-max para o intervalo [-1, 1]:


In [0]:
from sklearn.preprocessing import MinMaxScaler

# def min_max_normalize(train_df, test_df, features):
#     """Normaliza as features para o intervalo [-1, 1] sem data leakage"""
#     scaler = MinMaxScaler(feature_range=(-1, 1))
#     scaler.fit(train_df[features])  # Ajusta apenas no treino

#     # Aplica a transformação no treino e no teste
#     train_df[features] = scaler.transform(train_df[features])
#     test_df[features] = scaler.transform(test_df[features])

#     return train_df, test_df, scaler

# # Normalizando os dados
# features_to_normalize = selected_sensors
# df_train_raw, df_test, scaler = min_max_normalize(df_train_raw, df_test, features_to_normalize)

### Criando os RULs para Treino

O artigo usa um modelo de degradação linear por partes com RUL constante inicial (Re):


In [0]:
def create_rul_labels(df, Re, clip_at_zero=True):
    """Cria os rótulos RUL usando o modelo de degradação linear por partes"""
    grouped = df.groupby("engine_id")["cycle"].max().reset_index()
    grouped.columns = ["engine_id", "max_cycle"]

    df = df.merge(grouped, on="engine_id", how="left")
    df["RUL"] = df["max_cycle"] - df["cycle"]

    # Aplica o modelo de degradação linear por partes
    df["RUL"] = np.where(df["RUL"] > Re, Re, df["RUL"])

    return df.drop(columns=["max_cycle"])

# Usando Re=128 conforme sugerido no artigo para FD001
Re = 128
df_train_raw = create_rul_labels(df_train_raw, Re)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_rul_by_engine(df, engines_to_plot=5):
    """Plota o RUL ao longo do tempo para algumas engines"""
    # Selecionar engines para visualizar
    engine_ids = df["engine_id"].unique()[:engines_to_plot]

    plt.figure(figsize=(8, 6))

    for engine_id in engine_ids:
        engine_data = df[df["engine_id"] == engine_id]
        plt.plot(engine_data["cycle"], engine_data["RUL"], label=f'Engine {engine_id}')

    plt.xlabel('Cycle')
    plt.ylabel('RUL')
    plt.title('RUL por Engine ID')
    plt.legend()
    plt.grid(True)
    plt.show()


In [0]:
plot_rul_by_engine(df_train_raw, engines_to_plot=20)

### Preparando as Janelas Temporais

O artigo usa janelas temporais com tamanho (nw) e stride (ns):


separando os dados de validação e treino

In [0]:
from sklearn.model_selection import train_test_split

unique_engines = np.unique(df_train_raw['engine_id'])
train_engines, val_engines = train_test_split(unique_engines, test_size=0.2, shuffle=False)

df_train = df_train_raw[df_train_raw['engine_id'].isin(train_engines)]
df_val = df_train_raw[df_train_raw['engine_id'].isin(val_engines)]
display(train_engines, df_train.shape)
display(val_engines, df_val.shape)

In [0]:
def create_time_windows(df, window_size, window_stride, sensor_cols):
    """Cria janelas temporais dos dados dos sensores e retorna um DataFrame com info da janela"""
    sequences = []
    labels = []
    engine_ids = []
    last_cycles = []

    for engine_id in df["engine_id"].unique():
        engine_data = df[df["engine_id"] == engine_id]
        sensor_data = engine_data[sensor_cols].values
        rul_data = engine_data["RUL"].values

        # Cria janelas deslizantes
        for i in range(0, len(engine_data) - window_size + 1, window_stride):
            window = sensor_data[i: i + window_size]
            label = rul_data[i + window_size - 1]
            sequences.append(window)
            labels.append(label)
            engine_ids.append(engine_id)
            last_cycles.append(engine_data["cycle"].iloc[i + window_size - 1])

    # Flatten sequences for MLP input
    n_samples = len(sequences)
    n_timesteps = window_size
    n_features = len(sensor_cols)
    flattened_sequences = np.array(sequences).reshape((n_samples, n_timesteps * n_features))

    df_windows = pd.DataFrame({
        'engine_id': engine_ids,
        'last_cycle': last_cycles,
        'data_vector': list(flattened_sequences),
        'RUL': labels
    })

    return df_windows

In [0]:
# Parâmetros do artigo para FD001: nw=24, ns=1
window_size = 24
window_stride = 1

# Criando as sequências de treino
sensor_cols = selected_sensors
df_train_windows = create_time_windows(df_train, window_size, window_stride, sensor_cols)

# Criando as sequências de validação
df_val_windows = create_time_windows(df_val, window_size, window_stride, sensor_cols)

# Separando X_train e y_train
X_train = np.array(list(df_train_windows['data_vector']))
y_train = df_train_windows['RUL'].values

# Separando X_val e y_val
X_val = np.array(list(df_val_windows['data_vector']))
y_val = df_val_windows['RUL'].values
val_data= (X_val, y_val)


# Arquitetura da MLP

In [0]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Layer


def root_mean_squared_error(y_true, y_pred):
    # Converte os inputs para float32 explicitamente
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    squared_diff = tf.square(y_pred - y_true)
    mean_squared = tf.reduce_mean(squared_diff)
    rmse = tf.sqrt(mean_squared)
    return rmse


def rul_health_score(y_true, y_pred):
    # 1) Garante vetores 1D float32
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)

    d = y_pred - y_true

    score = tf.where(
        d < 0.0,
        tf.exp(-d / 13.0) - 1.0,
        tf.exp(d / 10.0) - 1.0
    )

    return tf.reduce_mean(score)


class MinMaxScalerLayer(Layer):
    def __init__(self, mins, denom, **kwargs):
        super().__init__(**kwargs)
        # Convert numpy arrays to TensorFlow tensors for saving
        self.mins = tf.constant(mins, dtype=tf.float32)
        self.denom = tf.constant(denom, dtype=tf.float32)


    def call(self, inputs):
        return 2.0 * (inputs - self.mins) / self.denom - 1.0

    def get_config(self):
        config = super().get_config()
        config.update({
            # Convert tensors back to numpy arrays for serialization
            "mins": self.mins.numpy().tolist(),
            "denom": self.denom.numpy().tolist(),
        })
        return config

In [0]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.metrics import MeanSquaredError, RootMeanSquaredError
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.regularizers import l1_l2


def create_mlp(input_dim, minmax_layer, lr=0.001, l1=0.1, l2=0.2):
    reg = l1_l2(l1=l1, l2=l2)

    model = Sequential(
        [
            Input(shape=(input_dim,)),
            minmax_layer,
            Dense(20, activation="relu",
                  kernel_regularizer=reg, bias_regularizer=reg),
            Dense(20, activation="relu",
                  kernel_regularizer=reg, bias_regularizer=reg),
            Dense(1, activation="linear",
                  kernel_regularizer=reg, bias_regularizer=reg),
        ],
        name="Arquitetura-1",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
        metrics=[
            MeanSquaredError(name="mse"),
            RootMeanSquaredError(name="rmse"),
            rul_health_score,
        ],
    )
    return model



In [0]:
# Criando o pre-processamento
mins  = X_train.min(axis=0).astype("float32")
maxs  = X_train.max(axis=0).astype("float32")
denom = maxs - mins
minmax_layer = MinMaxScalerLayer(mins, denom, name="minmax_-1_1")


# Criando o modelo
input_dim = X_train.shape[1]
model = create_mlp(input_dim=X_train.shape[1],
                   minmax_layer=minmax_layer)
model.summary()

In [0]:
history = model.fit(
    X_train, y_train,
    validation_data=val_data,
    epochs=100,
    batch_size=128,
    verbose=2
)

In [0]:
plots(history)

In [0]:
def plot_learning_curves(history):

    plt.figure(figsize=(5, 12))

    # Plot Loss (MSE)
    plt.subplot(3, 1, 1)
    plt.plot(history.history["loss"], label="Treino")
    plt.plot(history.history["val_loss"], label="Validação")
    plt.title("Loss (MSE)")
    plt.xlabel("Epochs")
    plt.ylabel("MSE")
    plt.legend()
    plt.grid(True)

    # Plot RMSE
    if "rmse" in history.history:
        plt.subplot(3, 1, 2)
        plt.plot(history.history["rmse"], label="Treino")
        plt.plot(history.history["val_rmse"], label="Validação")
        plt.title("RMSE")
        plt.xlabel("Epochs")
        plt.ylabel("RMSE")
        plt.legend()
        plt.grid(True)

    # Plot NASA Score
    if "rul_health_score" in history.history:
        plt.subplot(3, 1, 3)
        plt.plot(history.history["rul_health_score"], label="Treino")
        plt.plot(history.history["val_rul_health_score"], label="Validação")
        plt.title("NASA Score")
        plt.xlabel("Epochs")
        plt.ylabel("NASA Score")
        plt.legend()
        plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_learning_curves(history)

In [0]:
def rul_health_score_manual(y_true, y_pred):
    # Ensure inputs are numpy arrays and of float type
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    diffs = y_pred - y_true
    scores = np.where(diffs < 0, np.exp(- diffs / 13.0) - 1.0, np.exp(diffs / 10.0) - 1.0)
    return np.mean(scores)

In [0]:
import matplotlib.pyplot as plt

# Previsões e erros
y_pred_val = model.predict(X_val).flatten()
errors = y_val - y_pred_val

# Histograma dos erros
plt.figure()
plt.hist(errors, bins=50)
plt.title('Histograma dos Erros (y_true - y_pred)')
plt.xlabel('Erro')
plt.ylabel('Frequência')
plt.show()

# Calcula o RHS por amostra seguindo o paper
d = y_pred_val - y_val
rhs_samples = np.where(
    d < 0,
    np.exp(d / 13.0) - 1.0,   # under-prediction
    np.exp(d / 10.0) - 1.0    # over-prediction
)

# Plota histograma do RHS por amostra
plt.figure()
plt.hist(rhs_samples, bins=50)
plt.title('Histograma do RHS por Amostra')
plt.xlabel('RHS')
plt.ylabel('Frequência')
plt.show()

# Cálculo manual do RHS
rhs_val = rul_health_score_manual(y_val, y_pred_val)
print(f'RHS (Val Set): {rhs_val:.2f}')


In [0]:
def plot_rul_predictions(df, start=1,stop=1, dataset_type='train'):
    """
    Plota a comparação entre RUL real e RUL predito por engine_id.

    Parâmetros:
    - df: DataFrame contendo as colunas ['engine_id', 'cycle', 'RUL', 'RUL_PREDICTED']
    - engines_to_plot: Quantidade de engines a serem exibidas
    - dataset_type: 'train' ou 'test' (apenas para título)
    """
    engine_ids = df["engine_id"].unique()[start-1:stop]

    plt.figure(figsize=(5, 4))

    for engine_id in engine_ids:
        engine_data = df[df["engine_id"] == engine_id]
        plt.plot(engine_data["last_cycle"], engine_data["RUL"], label=f'Engine {engine_id} - Real', linestyle='-')
        plt.plot(engine_data["last_cycle"], engine_data["RUL_PREDICTED"], label=f'Engine {engine_id} - Predicted', linestyle='--')

    plt.xlabel('last_cycle')
    plt.ylabel('RUL')
    plt.title(f'Comparação RUL Real vs. Predito ({dataset_type.capitalize()} Set)')
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_prediction(df, engine_id= 1, dataset_type='train'):

    plt.figure(figsize=(5, 4))

    engine_data = df[df["engine_id"] == engine_id]
    plt.plot(engine_data["last_cycle"], engine_data["RUL"], label=f'Engine {engine_id} - Real', linestyle='-')
    plt.plot(engine_data["last_cycle"], engine_data["RUL_PREDICTED"], label=f'Engine {engine_id} - Predicted', linestyle='--')

    plt.xlabel('last_cycle')
    plt.ylabel('RUL')
    plt.title(f'Comparação RUL Real vs. Predito ({dataset_type.capitalize()} Set)')
    plt.legend()
    plt.grid(True)
    plt.show()

In [0]:
df_val_windows = df_val_windows.reset_index(drop=True)

# Criando o DataFrame de resultados
df_results = df_val_windows[["engine_id", "RUL", "last_cycle"]].copy()

df_results['RUL_PREDICTED'] = model.predict(X_val)

plot_rul_predictions(df_results, start=2,stop=3,dataset_type='val')
# plot_prediction(df=df_results, engine_id=1, dataset_type='val')

# Avaliando no conjunto de teste

In [0]:
df_rul['engine_id'] = df_rul.index + 1
display(df_rul.head())

### preparando os dados de teste

In [0]:
import numpy as np

# Parâmetros
window_size = 24
sensor_cols  = selected_sensors

# Supondo que df_rul tenha colunas ['engine_id', 'RUL']
df_rul = df_rul.rename(columns={'RUL':'true_RUL'})

# Geração das janelas e labels
X_test = []
y_test = []

for eid, group in df_test.groupby('engine_id', sort=False):
    data = group[sensor_cols].values
    # Pega só a última janela completa
    window = data[-window_size:]
    X_test.append(window.flatten())
    # Usa o RUL “verdadeiro” do df_rul
    y_test.append(
        df_rul.loc[df_rul['engine_id']==eid, 'true_RUL'].item()
    )

# Converte para arrays
X_test = np.stack(X_test)
y_test = np.array(y_test)


In [0]:
test_metrics = model.evaluate(X_test, y_test)
test_loss = test_metrics[0]
test_mse = test_metrics[1]
test_rmse = test_metrics[2]
test_rhs = test_metrics[3]

In [0]:
y_pred_test = model.predict(X_test).flatten()

plt.figure()
plt.hist(y_test, bins=30)
plt.title('Histograma do RUL Real (Teste)')
plt.xlabel('RUL')
plt.ylabel('Frequência')
plt.show()

plt.figure()
plt.hist(rhs_samples, bins=30)
plt.title('Histograma do RHS por Amostra (Teste)')
plt.xlabel('RHS')
plt.ylabel('Frequência')
plt.show()


In [0]:
# Create a DataFrame for plotting RUL and errors
df_test_results = pd.DataFrame({
    'engine_id': df_rul['engine_id'].values,  # Assuming df_rul is aligned with X_test/y_test
    'RUL_Actual': y_test,
    'RUL_Predicted': y_pred_test,
    'Prediction_Error': y_test - y_pred_test
})

# Plot 1: RUL vs. Engine ID
plt.figure(figsize=(12, 5))
plt.plot(df_test_results['engine_id'], df_test_results['RUL_Actual'], label='Actual RUL', marker='o', linestyle='-')
plt.plot(df_test_results['engine_id'], df_test_results['RUL_Predicted'], label='Predicted RUL', marker='x', linestyle='--')
plt.xlabel('Engine ID')
plt.ylabel('RUL')
plt.title('Actual vs. Predicted RUL on Test Set')
plt.legend()
plt.grid(True)
plt.show()


# Plot 2: Prediction Error vs. Engine ID
plt.figure(figsize=(12, 5))
plt.plot(df_test_results['engine_id'], df_test_results['Prediction_Error'])
plt.xlabel('Engine ID')
plt.ylabel('Prediction Error (Actual - Predicted)')
plt.title('Prediction Error on Test Set')
plt.axhline(0, color='red', linestyle='--', linewidth=0.8) # Add a horizontal line at 0 error
plt.grid(True)
plt.show()

# Salvando modelo

In [0]:
# Salvando o modelo
import json

save_dir = './turbofan_artifacts/architecture_1'
os.makedirs(save_dir, exist_ok=True)
print('Salvando artefatos em:', save_dir)

model.summary()
model_path = os.path.join(save_dir, 'turbofan_mlp_model.keras')
model.save(model_path)
print('Modelo salvo em:', model_path)

history_path = os.path.join(save_dir, 'history.json')
with open(history_path, 'w') as f:
    json.dump(history.history, f, indent=2)
print('History salvo em:', history_path)

# csv_path = os.path.join(save_dir, 'metrics.csv')
# df_cmp.to_csv(csv_path)

# print('Métricas salvas em:', csv_path)

In [0]:

# Check the contents of the save directory
!ls -lR {save_dir}

# Iterando para pegar a variancia do erro

In [0]:
df_all = create_time_windows(df_train_raw, window_size, window_stride, sensor_cols)
X_all = np.array(list(df_all['data_vector']))

y_all = df_all['RUL'].values

In [0]:
from keras.callbacks import EarlyStopping

In [0]:
def train_full_and_test(X_train_full, y_train_full, X_test, y_test, seed):
    mins  = X_train_full.min(axis=0).astype("float32")
    denom = (X_train_full.max(axis=0) - mins).astype("float32")
    minmax_layer = MinMaxScalerLayer(mins, denom)

    tf.keras.utils.set_random_seed(seed)

    model = create_mlp(input_dim=X_train_full.shape[1],
                       minmax_layer=minmax_layer)

    # usa 5 % do próprio treino para early-stopping
    model.fit(X_train_full, y_train_full,
              epochs=200, batch_size=128,
              callbacks=[EarlyStopping(patience=20,  restore_best_weights=True,
                                       monitor = "rmse")],
              verbose=0)

    metrics_dict = model.evaluate(X_test, y_test, verbose=0, return_dict=True)
    return metrics_dict["rmse"], metrics_dict["rul_health_score"]

# ------------------------------------------------------------------
# 3.  Executa para as mesmas seeds
# ------------------------------------------------------------------
rmse_test, rhs_test = [], []

for seed in [7, 14, 21, 42]:
    print(f"Training with seed: {seed}")
    rmse, rhs = train_full_and_test(X_all, y_all, X_test, y_test, seed)
    rmse_test.append(rmse)
    rhs_test.append(rhs)

rmse_test = np.array(rmse_test)
rhs_test  = np.array(rhs_test)

print(f"\nTESTE — RMSE 4 seeds = {rmse_test.mean():.2f} ± {rmse_test.std():.2f}")
print(f"TESTE — RHS  4 seeds = {rhs_test.mean():.2f} ± {rhs_test.std():.2f}")

In [0]:
rmse_test_mean = rmse_test.mean()
rmse_test_std = rmse_test.std()
rmse_test_min = rmse_test.min()
rmse_test_max = rmse_test.max()

print("RMSE (Test Set):")
print(f"  Average: {rmse_test_mean:.2f}")
print(f"  Standard Deviation: {rmse_test_std:.2f}")
print(f"  Minimum: {rmse_test_min:.2f}")
print(f"  Maximum: {rmse_test_max:.2f}")

rhs_test_mean = rhs_test.mean()
rhs_test_std = rhs_test.std()
rhs_test_min = rhs_test.min()
rhs_test_max = rhs_test.max()

print("\nRHS (Test Set):")
print(f"  Average: {rhs_test_mean:.2f}")
print(f"  Standard Deviation: {rhs_test_std:.2f}")
print(f"  Minimum: {rhs_test_min:.2f}")
print(f"  Maximum: {rhs_test_max:.2f}")